# 04 — Statistical Analysis & Figures (Cross-Polytope)

Combines the phase-1 (SimHash / E2LSH) results with the new
cross-polytope run to produce the protocol figures and outcome
classification (protocol section 12).


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
sns.set_theme(style='whitegrid')


In [ ]:
BENCHMARK_DIR = Path('../results/benchmark')
FIG_DIR = Path('../results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

def load_csvs(pattern):
    files = sorted(BENCHMARK_DIR.glob(pattern))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

df_quality = load_csvs('benchmark_quality_*seed*.csv')
df_timing = load_csvs('benchmark_timing_*seed*.csv')
df_cand = load_csvs('benchmark_candidates_*seed*.csv')
df_mem = load_csvs('benchmark_memory_*seed*.csv')


In [ ]:
merge_keys = ['dataset', 'base_size', 'split_seed', 'hash_seed', 'method']
df_merged = (df_quality.merge(df_timing, on=merge_keys)
                        .merge(df_cand, on=merge_keys)
                        .merge(df_mem, on=merge_keys))
df_agg = df_merged.groupby(['dataset', 'base_size', 'split_seed', 'method']).mean(numeric_only=True).reset_index()
print(df_agg.groupby(['dataset', 'base_size', 'method'])[['recall@10', 'median_latency_s', 'macro_f1']].mean())


### Statistical Tests
With only 5 splits, Wilcoxon p-values are interpreted cautiously.
We rely on absolute differences and 95% bootstrap CIs of the
paired difference vs linear.


In [ ]:
def bootstrap_ci(differences, n_bootstrap=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    boot = [rng.choice(differences, size=len(differences), replace=True).mean()
            for _ in range(n_bootstrap)]
    return float(np.percentile(boot, 100*alpha/2)), float(np.percentile(boot, 100*(1-alpha/2)))


In [ ]:
sig_rows = []
for ds in df_agg['dataset'].unique():
    for bs in df_agg[df_agg['dataset'] == ds]['base_size'].unique():
        sub = df_agg[(df_agg['dataset'] == ds) & (df_agg['base_size'] == bs)]
        lin = sub[sub['method'] == 'linear']
        for method in ['crosspolytope', 'SimHash', 'E2LSH']:
            m = sub[sub['method'] == method]
            if lin.empty or m.empty:
                continue
            diff = m['median_latency_s'].values - lin['median_latency_s'].values
            lo, hi = bootstrap_ci(diff)
            if len(diff) >= 5:
                _, p = stats.wilcoxon(diff)
            else:
                p = float('nan')
            sig_rows.append({'dataset': ds, 'base_size': bs, 'method': method,
                             'mean_diff_latency_s': float(np.mean(diff)),
                             'ci_low': lo, 'ci_high': hi, 'p_wilcoxon': p})
sig_df = pd.DataFrame(sig_rows)
if not sig_df.empty:
    sig_df.to_csv(BENCHMARK_DIR / 'significance_tests.csv', index=False)
    print(sig_df)


In [ ]:
# Figure 1: latency comparison
if not df_agg.empty:
    df_agg['ds_bs'] = df_agg['dataset'] + '_' + df_agg['base_size'].astype(str)
    plt.figure(figsize=(11, 6))
    sns.barplot(data=df_agg, x='ds_bs', y='median_latency_s', hue='method', errorbar=('ci', 95))
    plt.yscale('log')
    plt.title('Figure 1: Median query latency')
    plt.xticks(rotation=30)
    plt.tight_layout(); plt.savefig(FIG_DIR / 'fig1_latency.png', dpi=150)
    plt.show()


In [ ]:
# Figure 2: Recall@k
if not df_agg.empty:
    df_melt = df_agg.melt(id_vars=['ds_bs', 'method'], value_vars=['recall@1', 'recall@5', 'recall@10'])
    plt.figure(figsize=(11, 6))
    sns.boxplot(data=df_melt, x='ds_bs', y='value', hue='method')
    plt.title('Figure 2: Recall@k')
    plt.xticks(rotation=30)
    plt.tight_layout(); plt.savefig(FIG_DIR / 'fig2_recall.png', dpi=150)
    plt.show()


In [ ]:
# Figure 3: candidate fraction
if not df_cand.empty:
    df_c = df_cand[df_cand['method'] != 'linear']
    df_c = df_cand[df_cand['method'] == 'crosspolytope']
    if not df_c.empty:
        df_c['ds_bs'] = df_c['dataset'] + '_' + df_c['base_size'].astype(str)
        plt.figure(figsize=(11, 6))
        sns.boxplot(data=df_c, x='ds_bs', y='mean_cand_fraction')
        plt.title('Figure 3: Candidate fraction (cross-polytope)')
        plt.xticks(rotation=30)
        plt.tight_layout(); plt.savefig(FIG_DIR / 'fig3_candidates.png', dpi=150)
        plt.show()


In [ ]:
# Figure 4: build time
if not df_timing.empty:
    df_t = df_timing.copy()
    df_t['ds_bs'] = df_t['dataset'] + '_' + df_t['base_size'].astype(str)
    plt.figure(figsize=(11, 6))
    sns.barplot(data=df_t, x='ds_bs', y='build_time_s', hue='method', errorbar=('ci', 95))
    plt.title('Figure 4: Index build time')
    plt.xticks(rotation=30)
    plt.tight_layout(); plt.savefig(FIG_DIR / 'fig4_build.png', dpi=150)
    plt.show()


In [ ]:
# Figure 5: memory
if not df_mem.empty:
    df_m = df_mem.copy()
    df_m['ds_bs'] = df_m['dataset'] + '_' + df_m['base_size'].astype(str)
    df_m['mem_mb'] = df_m['index_memory_bytes'] / 1e6
    plt.figure(figsize=(11, 6))
    sns.barplot(data=df_m, x='ds_bs', y='mem_mb', hue='method', errorbar=('ci', 95))
    plt.title('Figure 5: Index memory (MB)')
    plt.xticks(rotation=30)
    plt.tight_layout(); plt.savefig(FIG_DIR / 'fig5_memory.png', dpi=150)
    plt.show()


In [ ]:
# Figure 6: recall-latency trade-off
if not df_agg.empty:
    plt.figure(figsize=(9, 6))
    sns.scatterplot(data=df_agg, x='median_latency_s', y='recall@10', hue='method', style='ds_bs', s=100)
    plt.xscale('log')
    plt.title('Figure 6: Recall@10 vs latency')
    plt.tight_layout(); plt.savefig(FIG_DIR / 'fig6_tradeoff.png', dpi=150)
    plt.show()


In [ ]:
# Figure 7: scaling (chest)
chest = df_agg[df_agg['dataset'] == 'chest']
if not chest.empty:
    plt.figure(figsize=(9, 6))
    chest['bs'] = chest['base_size'].astype(float)
    sns.lineplot(data=chest, x='bs', y='median_latency_s', hue='method', marker='o')
    plt.xscale('log'); plt.yscale('log')
    plt.title('Figure 7: Latency scaling (chest)')
    plt.tight_layout(); plt.savefig(FIG_DIR / 'fig7_scaling.png', dpi=150)
    plt.show()


In [ ]:
# Figure 8: hash-seed variance (cross-polytope)
df_cp = df_merged[df_merged['method'] == 'crosspolytope']
if not df_cp.empty:
    df_cp['ds_bs'] = df_cp['dataset'] + '_' + df_cp['base_size'].astype(str)
    plt.figure(figsize=(11, 6))
    sns.boxplot(data=df_cp, x='ds_bs', y='recall@10', hue='hash_seed')
    plt.title('Figure 8: Recall@10 by hash seed (cross-polytope)')
    plt.xticks(rotation=30)
    plt.tight_layout(); plt.savefig(FIG_DIR / 'fig8_hashseed.png', dpi=150)
    plt.show()


In [ ]:
# Outcome classification (protocol section 12)
outcomes = []
for ds in df_agg['dataset'].unique():
    for bs in df_agg[df_agg['dataset'] == ds]['base_size'].unique():
        sub = df_agg[(df_agg['dataset'] == ds) & (df_agg['base_size'] == bs)]
        lin = sub[sub['method'] == 'linear']
        if lin.empty:
            continue
        lin_lat = lin['median_latency_s'].mean()
        lin_f1 = lin['macro_f1'].mean()
        for method in ['crosspolytope']:
            m = sub[sub['method'] == method]
            if m.empty:
                continue
            r10 = m['recall@10'].mean()
            f1 = m['macro_f1'].mean()
            lat = m['median_latency_s'].mean()
            if r10 >= 0.95 and (lin_f1 - f1) <= 0.01:
                verdict = 'speedup' if lat < lin_lat * 0.9 else 'quality_ok_no_speedup'
            else:
                verdict = 'quality_fail'
            outcomes.append({'dataset': ds, 'base_size': bs, 'method': method, 'verdict': verdict,
                             'recall@10': r10, 'macro_f1': f1, 'latency_s': lat})
out_df = pd.DataFrame(outcomes)
out_df.to_csv(BENCHMARK_DIR / 'outcomes.csv', index=False)
print(out_df)


### Conclusion
Based on the outcomes classification: whether CrossPolytope LSH
provides a consistent speedup with recall >= 0.95, conditionally
works, or fails in quality. Results are compared against phase-1
(SimHash / E2LSH) in `lsh_sota_review.md` and `progress.md`.
